Primero nos autenticamos en Azure

In [7]:
# Importamos las clases necesarias de Azure ML
from azure.ai.ml import MLClient, load_component, dsl, Input
from azure.identity import DefaultAzureCredential
import os

# Creamos el cliente de Azure ML usando credenciales del entorno
# DefaultAzureCredential busca automáticamente: env vars, Azure CLI, VS Code, etc.
ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=os.getenv("AZURE_SUBSCRIPTION_ID"),    # ID de suscripción de Azure
    resource_group_name=os.getenv("AZURE_RESOURCE_GROUP"),  # Grupo de recursos
    workspace_name=os.getenv("AZURE_WORKSPACE_NAME")        # Nombre del workspace de ML
)

---

`load_component(source="./prep.yml")` 
- lee el archivo `prep.yml` 
- lo convierte en un **objeto Python invocable**.

---

### prep.yml
>```yaml
>name: prep_data
>inputs:
>  input_data: uri_file       # Espera un archivo como entrada
>outputs:
>  output_data: uri_file      # Produce un archivo como salida
>command: python prep.py 
>    --input_data ${{inputs.input_data}} 
>    --output_data ${{outputs.output_data}}
>```
---

### prep.py 

>```python
>parser.add_argument("--input_data", dest='input_data', type=str)   
>    # Recibe la ruta del archivo de entrada
>
>parser.add_argument("--output_data", dest='output_data', type=str) 
>    # Recibe la ruta donde guardar la salida
>
>df = pd.read_csv(args.input_data)                                   
>    # Lee el CSV desde la ruta recibida
>
># ------- limpieza y normalización de datos ---------#
>
>output_df = df.to_csv((Path(args.output_data) / "prepped-data.csv")) 
>    # Guarda el resultado en la ruta de salida
>```

---

### Flujo de datos

```text
prep.yml (declaración) ── load_component──►  prep_component (objeto Python)
                                                    │
prep.py (ejecución real) ◄── Azure ML ejecuta ──────┘
         el script en la nube con las rutas reales
```

**Importante:** `prep_component` NO ejecuta `prep.py` aquí. Solo **describe** qué hacer. La ejecución real ocurre en Azure cuando se envía el pipeline.

In [8]:
# crea obj Python invocable a partir del archivo YAML
# Este objeto sabe que acepta "input_data" y expone "outputs.output_data"
prep_component = load_component(source="./prep.yml")

---

### @dsl.pipeline

es un **DECORADOR** de Azure ML. Coge la función que está debajo y la **reemplaza** por un objeto Pipeline.


### Los argumentos del decorador

| Argumento | Significado | Equivalente en YAML |
|---|---|---|
| `compute="cpu-cluster"` | Dónde ejecutar los pasos | `compute: azureml:cpu-cluster` |
| `description="..."` | Descripción del pipeline | `description: "..."` |

### El parámetro de la función

```python
def pipeline_de_preparacion(datos_crudos_entrada):
```

`datos_crudos_entrada` es un **input del pipeline**. Cuando luego llamamos:

```python
pipeline_job = pipeline_de_preparacion(
    datos_crudos_entrada=Input(
        type="uri_file",
        path=ruta_datos_azure)
)
```

Le estamos diciendo: "este pipeline recibirá un archivo (`uri_file`) ubicado en esta ruta de Azure".

**Nota:** La función ya no es una función normal. Al estar decorada, `pipeline_de_preparacion(...)` genera un objeto `PipelineJob`, no ejecuta código Python.

---

## Cuerpo del pipeline — Conectar componentes

Dentro de la función definimos **cómo se conectan los pasos** del pipeline.

### 1. Definir la función con sus inputs

```python
def pipeline_de_preparacion(datos_crudos_entrada):
```

- `datos_crudos_entrada` → es el **input del pipeline** (un archivo CSV crudo)
- Este nombre lo usaremos dentro para pasárselo a los componentes

### 2. Invocar un componente

```python
paso_limpieza = prep_component(input_data=datos_crudos_entrada)
```

- `prep_component` → el objeto Python creado por `load_component` desde `prep.yml`
- `input_data=` → es el nombre del parámetro **definido en prep.yml** (`inputs.input_data`)
- `datos_crudos_entrada` → es el input del pipeline que le pasamos al componente
- `paso_limpieza` → es un **objeto que representa la ejecución de ese paso**, no el resultado real

**No hay ejecución real aquí.** Esto solo construye un grafo: "paso_limpieza ejecuta prep_component con tal entrada".

### 3. Devolver las salidas mapeadas

```python
return {
    "datos_limpios": paso_limpieza.outputs.output_data
}
```

- `"datos_limpios"` → **nombre que inventamos** para la salida del pipeline
- `paso_limpieza.outputs.output_data` → referencia a la salida del componente (`output_data` viene del YAML)
- El `return` aquí **no es un return de Python real**. Es sintaxis especial que el decorador interpreta para construir la definición del pipeline.

### Traducción a YAML de lo que realmente genera:

```yaml
outputs:
  datos_limpios:            # ← el nombre del return
    type: uri_file
    value: ${{jobs.paso_limpieza.outputs.output_data}}
```

## Código completo del pipeline (con anotaciones)

```python
@dsl.pipeline(
    compute="cpu-cluster",                                    # Dónde ejecutar (clúster de Azure)
    description="Pipeline para limpiar y normalizar datos"    # Descripción del pipeline
)
def pipeline_de_preparacion(datos_crudos_entrada):            # Input del pipeline (uri_file)
    
    # Llamamos al componente cargado desde prep.yml
    # input_data es el nombre del input definido en prep.yml
    paso_limpieza = prep_component(input_data=datos_crudos_entrada)
    
    # Mapeamos las salidas del pipeline
    # "datos_limpios" es el nombre que le damos a la salida
    # paso_limpieza.outputs.output_data es la salida del componente (definida en prep.yml)
    return {
        "datos_limpios": paso_limpieza.outputs.output_data
    }
```

### Llamada al pipeline (fuera de la función):

```python
ruta_datos_azure = "azureml://datastores/workspaceblobstore/paths/datos_crudos.csv"

pipeline_job = pipeline_de_preparacion(
    datos_crudos_entrada=Input(type="uri_file", path=ruta_datos_azure)
    #   ↑ nombre del parámetro       ↑ tipo: archivo   ↑ ruta en Azure
)

# Enviar a Azure
pipeline_enviado = ml_client.jobs.create_or_update(
    pipeline_job,
    experiment_name="limpieza_datos_personalizado"
)
```

### Resumen del flujo completo:

```
1. prep.yml  ──load_component──►  prep_component (objeto Python)
2. @dsl.pipeline decora la función pipeline_de_preparacion
3. pipeline_de_preparacion(...) genera un PipelineJob (no ejecuta código)
4. ml_client.jobs.create_or_update(...) envía el pipeline a Azure
5. Azure ML ejecuta prep.py en cpu-cluster con los datos reales
```

In [10]:
# Definimos la ruta del archivo de entrada que está en tu Datastore de Azure
ruta_datos_azure = "azureml://datastores/workspaceblobstore/paths/datos_crudos.csv"

# el trabajo del pipeline recibe como input el archivo de datos crudos 
# y lo pasa al componente de limpieza
pipeline_job = pipeline_de_preparacion(
    datos_crudos_entrada=Input(
        type="uri_file", 
        path=ruta_datos_azure)
)

# Enviamos el Job al Workspace de Azure
pipeline_enviado = ml_client.jobs.create_or_update(
    pipeline_job, 
    experiment_name="limpieza_datos_personalizado"
)

print(f"¡Trabajo enviado con éxito!")
print(f"Puedes seguir el progreso aquí: {pipeline_enviado.studio_url}")

NameError: name 'pipeline_de_preparacion' is not defined

---
Guardar el componente para poder reutilizarlo en otros pipelines:

In [ ]:
registered_component = ml_client.components.create_or_update(pipeline_job)